In [27]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.linear_model import LinearRegression

In [28]:
data = pd.read_csv('/content/apple_stock_data.csv')
print(data.head(10))

         Date     Close     Volume      Open      High       Low
0  02/28/2020   $273.36  106721200   $257.26   $278.41   $256.37
1  02/27/2020   $273.52   80151380    $281.1      $286   $272.96
2  02/26/2020   $292.65   49678430   $286.53   $297.88    $286.5
3  02/25/2020   $288.08   57668360   $300.95   $302.53   $286.13
4  02/24/2020   $298.18   55548830   $297.26   $304.18   $289.23
5  02/21/2020   $313.05   32426420   $318.62   $320.45    $310.5
6  02/20/2020    $320.3   25141490   $322.63   $324.65   $318.21
7  02/19/2020   $323.62   23495990      $320   $324.57      $320
8  02/18/2020      $319   38190550   $315.36   $319.75   $314.61
9  02/14/2020   $324.95   20028450   $324.74   $325.98   $322.85


In [29]:
#Since it's stock data, I'll convert the date to datetime, set it as the index, and use the Close price.

data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)
data= data[[' Close']]


In [30]:
data.head()

,Close
Date,
2020-02-28,$273.36
2020-02-27,$273.52
2020-02-26,$292.65
2020-02-25,$288.08
2020-02-24,$298.18


In [31]:
#Close price data between 0 and 1 using MinMaxScaler to ensure compatibility with the LSTM model.
scaler = MinMaxScaler(feature_range=(0, 1))
data[' Close'] = data[' Close'].str.replace('$', '', regex=False).str.strip().astype(float)
data[' Close'] = scaler.fit_transform(data[[' Close']])

/tmp/ipython-input-31-1351283040.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[' Close'] = data[' Close'].str.replace('$', '', regex=False).str.strip().astype(float)
/tmp/ipython-input-31-1351283040.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[' Close'] = scaler.fit_transform(data[[' Close']])


In [32]:
#Now, let’s prepare the data for LSTM by creating sequences of a defined length (e.g., 60 days) to predict the next day’s price.
def create_sequences(data, seq_length=60):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 60
X, y = create_sequences(data[' Close'].values, seq_length)

In [33]:
# We will split the sequences into training and test sets ( 80% training, 20% testing).

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [34]:
# We will build a sequential LSTM model with layers to capture the temporal dependencies in the data.

lstm_model = Sequential()
lstm_model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
lstm_model.add(LSTM(units=50))
lstm_model.add(Dense(1))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [35]:
# We’ll compile the model with a suitable optimizer and loss, then fit it to the training data.

lstm_model.compile(optimizer='adam', loss='mean_squared_error')
lstm_model.fit(X_train, y_train, epochs=20, batch_size=32)

Epoch 1/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - loss: 0.0227
Epoch 2/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - loss: 3.3921e-04
Epoch 3/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - loss: 3.2375e-04
Epoch 4/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - loss: 2.9773e-04
Epoch 5/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - loss: 2.8801e-04
Epoch 6/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 3.2153e-04
Epoch 7/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - loss: 3.4033e-04
Epoch 8/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 2.5898e-04
Epoch 9/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - loss: 3.0762e-04
Epoch 10/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - loss: 2.5604e-04
Epoch 11/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - loss: 2.6317e-04
Epoch 12/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - loss: 2.0613e-04
Epoch 13/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 6s 70ms/step - loss: 1.8891e-04
Epoch 14/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 2.3145e-04
Epoch 15/20
62/62 ━

In [36]:
# Now, let’s train the second model by creating 3-day lag features for Linear Regression.

data['Lag_1'] = data[' Close'].shift(1)
data['Lag_2'] = data[' Close'].shift(2)
data['Lag_3'] = data[' Close'].shift(3)
data = data.dropna()

In [37]:
# we will split the data accordingly for training and testing.

X_lin = data[['Lag_1', 'Lag_2', 'Lag_3']]
y_lin = data[' Close']
X_train_lin, X_test_lin = X_lin[:train_size], X_lin[train_size:]
y_train_lin, y_test_lin = y_lin[:train_size], y_lin[train_size:]

In [38]:
# Let’s train the linear regression model

from sklearn.linear_model import LinearRegression
lin_model = LinearRegression()
lin_model.fit(X_train_lin, y_train_lin)

LinearRegression()

In [39]:
# Here’s how to make predictions using LSTM on the test set and inverse transform the scaled predictions.
X_test_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
lstm_predictions = lstm_model.predict(X_test_lstm)
lstm_predictions = scaler.inverse_transform(lstm_predictions)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step


In [40]:
# Here’s how to generate predictions using Linear Regression and inverse-transform them.
lin_predictions = lin_model.predict(X_test_lin)
lin_predictions = scaler.inverse_transform(lin_predictions.reshape(-1, 1))

In [41]:
min_len = min(len(lstm_predictions), len(lin_predictions))
hybrid_predictions = (
    0.7 * lstm_predictions[:min_len] +
    0.3 * lin_predictions[:min_len]
)


## Predicting using the Hybrid Model

In [42]:
# Let’s see how to make predictions for the next 10 days using our hybrid model.
# Here’s how to predict the Next 10 Days using LSTM

lstm_future_predictions = []
last_sequence = X[-1].reshape(1, seq_length, 1)
for _ in range(10):
    lstm_pred = lstm_model.predict(last_sequence)[0, 0]
    lstm_future_predictions.append(lstm_pred)
    lstm_pred_reshaped = np.array([[lstm_pred]]).reshape(1, 1, 1)
    last_sequence = np.append(last_sequence[:, 1:, :], lstm_pred_reshaped, axis=1)
lstm_future_predictions = scaler.inverse_transform(np.array(lstm_future_predictions).reshape(-1, 1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


In [43]:
#Here’s how to predict the Next 10 Days using Linear Regression

recent_data = data[' Close'].values[-3:]
lin_future_predictions = []
for _ in range(10):
    lin_pred = lin_model.predict(recent_data.reshape(1, -1))[0]
    lin_future_predictions.append(lin_pred)
    recent_data = np.append(recent_data[1:], lin_pred)
lin_future_predictions = scaler.inverse_transform(np.array(lin_future_predictions).reshape(-1, 1))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/

In [44]:
# Here’s how to combine the predictive power of both models to make predictions for the next 10 days

hybrid_future_predictions = (0.7 * lstm_future_predictions) + (0.3 * lin_future_predictions)

In [45]:
# Here’s how to create the final DataFrame to look at the predictions

future_dates = pd.date_range(start=data.index[-1] + pd.Timedelta(days=1), periods=10)
predictions_df = pd.DataFrame({
    'Date': future_dates,
    'LSTM Predictions': lstm_future_predictions.flatten(),
    'Linear Regression Predictions': lin_future_predictions.flatten(),
    'Hybrid Model Predictions': hybrid_future_predictions.flatten()
})
print(predictions_df)

        Date  LSTM Predictions  Linear Regression Predictions  \
0 2010-03-02         32.644154                      30.075606   
1 2010-03-03         32.556404                      30.014586   
2 2010-03-04         32.589912                      30.018833   
3 2010-03-05         32.706161                      30.245627   
4 2010-03-06         32.877701                      30.193550   
5 2010-03-07         33.085430                      30.181808   
6 2010-03-08         33.316326                      30.414333   
7 2010-03-09         33.561684                      30.372544   
8 2010-03-10         33.815777                      30.344701   
9 2010-03-11         34.074902                      30.581702   

   Hybrid Model Predictions  
0                 31.873588  
1                 31.793858  
2                 31.818589  
3                 31.968000  
4                 32.072456  
5                 32.214344  
6                 32.445728  
7                 32.604940  
8             